# 04: Experimental Manipulation Functions

The four functions implementing the factorial manipulations:

1. apply_imbalance: exact class ratios (50/50, 80/20, 90/10, 95/5)
2. apply_horizontal_minimization: stratified row reduction (100/75/50/25%)
3. apply_vertical_minimization: MI-ranked feature reduction (100/75/50/25%)
4. make_split: stratified 70/30 train/test partition

All functions are seed-deterministic (verified below); the five study seeds are fixed in the design grid. Exported to
manipulation_functions.py for import by downstream notebooks.


## Order of operations and base sample size

**Pipeline order: imbalance -> minimization -> split -> metrics on test.**
Rationale: Current research frames minimization policies as acting on an already imbalanced data ecosystems; this manipulation order mirrors that causal framing.

**Common base size: base_n = 15,000 for all conditions.** All imbalance ratios are constructed by undersampling to this fixed total, so the imbalance factor is not confounded with sample size. The cap is set by Adult's ~7,840 positives, which must supply 50% of base_n in the 50/50 condition. A consequence: the 50/50 condition discards the most majority data; this is the standard cost of a controlled-ratio design.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif

BUCKET = "osilesi-dissertation-data-2026"
adult = pd.read_csv(f"s3://{BUCKET}/processed/adult_final_pruned.csv")
print(adult.shape, adult["target"].mean().round(3))

(32561, 43) 0.241


In [2]:
RATIOS = {"50/50": 0.50, "80/20": 0.20, "90/10": 0.10, "95/5": 0.05}

def apply_imbalance(df, ratio_label, seed, base_n=20000):
    """Undersample to an exact class ratio at a fixed total size.

    ratio_label: one of '50/50', '80/20', '90/10', '95/5'
    base_n: common total size across all conditions so ratio is
            not confounded with sample size.
    """
    minority_frac = RATIOS[ratio_label]
    n_min = int(base_n * minority_frac)
    n_maj = base_n - n_min

    minority = df[df["target"] == 1]
    majority = df[df["target"] == 0]

    rng_min = minority.sample(n=n_min, random_state=seed)
    rng_maj = majority.sample(n=n_maj, random_state=seed)

    out = pd.concat([rng_min, rng_maj]).sample(frac=1, random_state=seed)
    return out.reset_index(drop=True)

In [4]:
for label in RATIOS:
    d = apply_imbalance(adult, label, seed=42, base_n=15000)
    print(label, d.shape[0], round(d["target"].mean(), 3))
# expect 15000 rows each, minority share 0.50 / 0.20 / 0.10 / 0.05

50/50 15000 0.5
80/20 15000 0.2
90/10 15000 0.1
95/5 15000 0.05


## Stratification in horizontal minimization

Row reduction is stratified on target x sex_female so both the minority class and the female subgroup survive proportionally even at 25% retention layered on 95/5 imbalance. The unit test below records the female-positive count in the harshest cell as evidence that fairness metrics remain computable across all 960 runs.


In [6]:
LEVELS = {"100": 1.00, "75": 0.75, "50": 0.50, "25": 0.25}

def apply_horizontal_minimization(df, level_label, seed):
    """Retain a stratified fraction of rows.

    Stratification on target x sex_female preserves class and
    subgroup proportions under aggressive reduction.
    """
    frac = LEVELS[level_label]
    if frac == 1.00:
        return df.copy()

    strata = df["target"].astype(str) + "_" + df["sex_female"].astype(str)
    out = (df.groupby(strata, group_keys=False)
             .apply(lambda g: g.sample(frac=frac, random_state=seed)))
    return out.sample(frac=1, random_state=seed).reset_index(drop=True)

In [7]:
harsh = apply_imbalance(adult, "95/5", seed=42, base_n=15000)
for label in LEVELS:
    d = apply_horizontal_minimization(harsh, label, seed=42)
    n_pos_f = ((d["target"] == 1) & (d["sex_female"] == 1)).sum()
    print(label, d.shape[0], round(d["target"].mean(), 3),
          "female positives:", n_pos_f)

100 15000 0.05 female positives: 106
75 11251 0.05 female positives: 80
50 7500 0.05 female positives: 53
25 3749 0.05 female positives: 26


## Vertical minimization design choices

- Ranking criterion: mutual information with the target (informed minimization: organizations remove what they judge least useful, the realistic policy scenario). 
  
- `target` and `sex_female` are never removal candidates: the fairness attribute is measurement infrastructure, not a minimizable feature; removing it would make RQ2 uncomputable.
- discrete_features=True: appropriate since post-encoding the matrix is overwhelmingly binary; applied uniformly for consistency.
  

In [8]:
def apply_vertical_minimization(df, level_label, seed):
    """Retain the top fraction of features ranked by mutual
    information with the target. target and sex_female are
    always retained and excluded from ranking.
    """
    frac = LEVELS[level_label]
    protected = ["target", "sex_female"]
    if frac == 1.00:
        return df.copy()

    feats = [c for c in df.columns if c not in protected]
    mi = mutual_info_classif(df[feats], df["target"],
                             discrete_features=True, random_state=seed)
    ranking = pd.Series(mi, index=feats).sort_values(ascending=False)

    n_keep = max(1, int(len(feats) * frac))
    keep = ranking.head(n_keep).index.tolist()
    return df[keep + protected].copy()

In [9]:
base = apply_imbalance(adult, "80/20", seed=42, base_n=15000)
for label in LEVELS:
    d = apply_vertical_minimization(base, label, seed=42)
    print(label, "features:", d.shape[1] - 2,
          "sex_female present:", "sex_female" in d.columns)

100 features: 41 sex_female present: True
75 features: 30 sex_female present: True
50 features: 20 sex_female present: True
25 features: 10 sex_female present: True


## Split design

70/30 stratified split on target x sex_female. At base_n = 15,000 the test partition is 4,500 rows, sufficient for stable subgroup TPR/FNR estimation in every condition (verified in the unit test below).


In [10]:
def make_split(df, seed, test_size=0.30):
    strata = df["target"].astype(str) + "_" + df["sex_female"].astype(str)
    train, test = train_test_split(df, test_size=test_size,
                                   stratify=strata, random_state=seed)
    return train.reset_index(drop=True), test.reset_index(drop=True)

In [11]:
# --- Unit test: make_split on the harshest condition ---

# Build the worst case: 95/5 imbalance, then split
harsh = apply_imbalance(adult, "95/5", seed=42, base_n=15000)
train, test = make_split(harsh, seed=42)

def profile(df, label):
    return {
        "partition": label,
        "rows": len(df),
        "pct_positive": round(df["target"].mean() * 100, 2),
        "pct_female": round(df["sex_female"].mean() * 100, 2),
        "n_female_positives": int(((df["target"] == 1) &
                                   (df["sex_female"] == 1)).sum()),
    }

report = pd.DataFrame([
    profile(harsh, "input (pre-split)"),
    profile(train, "train"),
    profile(test,  "test"),
])
report

,partition,rows,pct_positive,pct_female,n_female_positives
0,input (pre-split),15000,5.0,37.77,106
1,train,10500,5.0,37.77,74
2,test,4500,5.0,37.78,32


In [12]:
t1a, t1b = make_split(harsh, seed=42)
t2a, t2b = make_split(harsh, seed=42)
assert t1a.equals(t2a) and t1b.equals(t2b), "make_split not deterministic"
print("make_split deterministic: PASS")

make_split deterministic: PASS


In [13]:
# --- Integration test: full chain for one arbitrary condition ---
# Condition: 90/10 imbalance, 50% vertical minimization, seed 7
# Order of operations: imbalance -> minimization -> split

# Step 1: apply imbalance
d = apply_imbalance(adult, "90/10", seed=7, base_n=15000)
print("after imbalance:     ", d.shape,
      "| pct positive:", round(d["target"].mean() * 100, 1))

# Step 2: apply minimization (vertical in this rehearsal)
d = apply_vertical_minimization(d, "50", seed=7)
print("after minimization:  ", d.shape,
      "| sex_female present:", "sex_female" in d.columns)

# Step 3: split
train, test = make_split(d, seed=7)
print("train:", train.shape, "| test:", test.shape)
print("test pct positive:", round(test["target"].mean() * 100, 1),
      "| test female positives:",
      int(((test["target"] == 1) & (test["sex_female"] == 1)).sum()))

after imbalance:      (15000, 43) | pct positive: 10.0
after minimization:   (15000, 22) | sex_female present: True
train: (10500, 22) | test: (4500, 22)
test pct positive: 10.0 | test female positives: 65


## Integration test: PASSED [2026-09-01]

Full chain (imbalance -> minimization -> split) verified on 90/10 /50% / vertical / seed 7, and rerun with horizontal minimization.
Identical output on repeated execution confirms composed-chain determinism. Functions exported to manipulation_functions.py and archived to /code.


In [14]:
from manipulation_functions import *
print(RATIOS, LEVELS)

{'50/50': 0.5, '80/20': 0.2, '90/10': 0.1, '95/5': 0.05} {'100': 1.0, '75': 0.75, '50': 0.5, '25': 0.25}
